# 03 · Walking access and facility siting

**Question:** Which households are poorly served once streets and barriers are included?

**Runtime:** Python / Pyodide in JupyterLite. Choose **Run → Run All Cells**.
Each notebook is independent; start with a fresh kernel. Initial runtime/package downloads need internet.
The initial baseline uses **synthetic educational fixtures** near a Gaborone-area anchor.
Extended investigations explicitly identify bundled real OSM, Sentinel or terrain extracts where used;
these do not validate the synthetic population, mobility, exposure or scenario assumptions.

**Astra experiment:** Ask Astra to choose a new clinic using the exported candidate scores, then check its answer against exhaustive search.
Run the numerical baseline first, then give Astra the exported evidence and the prompt below.
The baseline does not call or run GPT-6 Astra. No API key is needed.


In [ ]:
import sys, json, math, time
from pathlib import Path
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from twin import *
style()
print("Runtime:", sys.platform, "| data: synthetic | seed: 42")


## A barrier makes distance a network problem
The vertical barrier has one northern crossing. Building-to-node connectors are straight-line approximations.
All streets are bidirectional; there is no timetable, capacity or clinical service model.


In [ ]:
c=city(); pop=allocate(c['capacity'],1200)
barrier=[(r*11+5,r*11+6) for r in range(10)]
nodes,graph=network(blocked=barrier)
clinic=22; speed_m_min=80.; threshold_min=15.
idx,connector=snap(c['xy'],nodes)
base=(shortest(graph,[clinic])[idx]+connector)/speed_m_min
euclidean=np.linalg.norm(c['xy']-nodes[clinic],axis=1)/speed_m_min
assert (base+1e-9>=euclidean).all()
scores=[]
for candidate in range(len(nodes)):
    times=(shortest(graph,[clinic,candidate])[idx]+connector)/speed_m_min
    scores.append(float(np.average(times,weights=pop)))
best=int(np.argmin(scores))
improved=(shortest(graph,[clinic,best])[idx]+connector)/speed_m_min
assert (improved<=base+1e-10).all()
print('Best new node:',best,'| weighted minutes:',np.average(base,weights=pop),'->',scores[best])
print('Residents within threshold:',pop[base<=threshold_min].sum(),'->',pop[improved<=threshold_min].sum())


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,5))
for ax,times,label in zip(axes,[base,improved],['Baseline','With one additional clinic']):
    for u,edges in graph.items():
        for v,w in edges:
            if u<v: ax.plot(*nodes[[u,v]].T,color='#bcc8ce',lw=.6)
    dots=ax.scatter(*c['xy'].T,c=times,vmin=0,vmax=base.max(),cmap='magma_r',s=20)
    ax.scatter(*nodes[clinic],marker='P',c='cyan',s=140)
    if label!='Baseline': ax.scatter(*nodes[best],marker='*',c='lime',s=160)
    map_axes(ax,label)
fig.colorbar(dots,ax=list(axes),label='Walking minutes'); plt.show()
export_json('03_access.json',dict(synthetic=True,candidate_weighted_minutes=scores,threshold_min=threshold_min,
            objective='minimize population-weighted mean walking minutes',candidate_id='array index'))


## Teaching lab: Choose a clinic
**Before using AI:** write a prediction, run the numerical baseline, and explain one surprising result.

**Astra evidence:** send only 03_access.json; the objective is lowest population-weighted mean travel time. Use a fresh conversation and record the exact model,
date, interface, tools and prompt. A task with a supplied result tests interpretation, not independent calculation.

**Copyable Astra prompt:**
> Return the candidate node with the minimum weighted minutes, its score, and one limitation of the walking model. Candidate array index is node ID.

**Implementation brief — review the runnable extension below:**
> Add a service-threshold sensitivity plot at 10, 15 and 20 minutes for baseline and the chosen additional clinic. Edit the authoring source in `scripts/make_notebooks.py` (or
> `scripts/teaching_labs.py` for labs 11–13), regenerate the notebooks and preserve browser compatibility.
> Use NumPy, Matplotlib and the standard library. Include a small reference case that can be checked by hand.

**Acceptance evidence:** Coverage is nondecreasing with threshold; adding a clinic never increases anyone’s shortest travel time.

The extended investigation later in this notebook implements this line of work with maps and checks.
Use the brief to review the implementation or ask Codex for a further controlled variation.

**Share back:** save your original prediction, changed parameter, result, model response and one limitation.
Download the edited notebook and exports. If you have no model access, exchange explanations with a partner
and evaluate them with the same criteria; do not label that activity as an Astra trial.


## Extended investigation: Access depends on mode, hours, waiting and capacity

Compare walking, slower accessible travel, a closed facility, and a six-minute transfer wait. An inaccessible crossing disconnects one part of the accessible network. A shortest-pair greedy allocation respects clinic capacities and explicitly records unserved residents; it is a transparent heuristic, not an optimal service allocation. The transfer wait is an invented scenario, not an observed combi schedule.

**Read the concept map first:** identify the input, method, evidence and limit. Predict the spatial pattern before running the comparison.


In [ ]:
from extension_tools import concept_map, export_investigation
concept_map('Access depends on mode, hours, waiting and capacity', ['Street graph + residents', 'Mode / opening / capacity', 'Unserved population map', 'Greedy ≠ optimal assignment'])
plt.show()


In [ ]:
from extension_tools import capacity_assignment
ex_c=city(); ex_pop=allocate(ex_c['capacity'],1200)
ex_barrier=[(r*11+5,r*11+6) for r in range(10)]
ex_nodes,ex_walk=network(blocked=ex_barrier); _,ex_access=network(blocked=ex_barrier+[(115,116)])
ex_idx,ex_connector=snap(ex_c['xy'],ex_nodes); ex_clinics=[22,94]
ex_modes={}; ex_served={}; ex_unserved={}
for label,graph,speed,wait,opened in [('Walk',ex_walk,80.,0.,[True,True]),('Accessible',ex_access,55.,0.,[True,True]),('Late / east closed',ex_access,55.,0.,[True,False]),('Transfer wait',ex_walk,80.,6.,[True,True])]:
    times=np.column_stack([(shortest(graph,[node])[ex_idx]+ex_connector)/speed+wait if is_open else np.full(144,np.inf) for node,is_open in zip(ex_clinics,opened)])
    allocation,unserved=capacity_assignment(times,ex_pop,[600,400])
    ex_modes[label]=times; ex_served[label]=allocation; ex_unserved[label]=unserved
    assert allocation.sum()+unserved.sum()==1200
fig,axes=plt.subplots(1,3,figsize=(14,4))
labels=list(ex_modes); axes[0].barh(labels,[ex_unserved[k].sum() for k in labels],color='#c44e52')
axes[0].set(title='Capacity + disconnection + hours',xlabel='Unserved residents')
for label in labels:
    nearest=ex_modes[label].min(axis=1)
    coverage=[float(ex_pop[nearest<=limit].sum()/1200) for limit in [5,10,15,20,30]]
    assert (np.diff(coverage)>=0).all()
    axes[1].plot([5,10,15,20,30],coverage,marker='o',label=label)
ex_original_coverage={}
for label,travel in [('Baseline: 1 clinic',base),('Chosen addition',improved)]:
    values=[float(pop[travel<=limit].sum()/pop.sum()) for limit in [5,10,15,20,30]]
    assert (np.diff(values)>=0).all()
    ex_original_coverage[label]=values
    axes[1].plot([5,10,15,20,30],values,ls='--',label=label)
assert (improved<=base+1e-10).all()
axes[1].set(xlabel='Travel threshold (min)',ylabel='Fraction of all residents',title='Potential access before capacity'); axes[1].legend(fontsize=7)
dots=axes[2].scatter(*ex_c['xy'].T,c=ex_unserved['Late / east closed'],s=30,cmap='magma'); map_axes(axes[2],'Late-session unserved residents'); fig.colorbar(dots,ax=axes[2],label='People')
fig.tight_layout(); extension_figure=fig
extension_evidence=dict(population=1200,capacity=[600,400],unserved_by_mode={k:int(v.sum()) for k,v in ex_unserved.items()},original_clinic_coverage=ex_original_coverage,thresholds_min=[5,10,15,20,30],assumptions='Greedy shortest-pair allocation; synthetic schedules and travel modes')
extension_expected=dict(capacity_shortfall=200,worst_mode=max(ex_unserved,key=lambda k:ex_unserved[k].sum()))
extension_task='Compute the minimum shortfall from nominal capacity alone and name the scenario with the largest unserved total. Return capacity_shortfall and worst_mode. Explain no extra claims in the JSON.'


In [ ]:
extension_prompt,extension_reference=export_investigation('03',extension_evidence,extension_expected,extension_task,extension_figure)
plt.show()
print('Send only 03_extension_prompt.json and optionally 03_extension_map.png to Astra. Keep the reference in this kernel.')


### Ask, check and explain the spatial result
Download `03_extension_prompt.json` and `03_extension_map.png`. Send the evidence in a fresh Astra
session with the exact task and output fields. For a separate image-only experiment, supply only the image
and task wording, record that condition, and allow clarification if the image does not contain enough information.
Never send the reference or this complete notebook for a blind trial.

Paste the unedited response below or upload a UTF-8 response file. The default is deliberately empty:
**no model trial has run**. After recording a real answer, set `extension_source='MODEL'` and complete the metadata.
The numeric tolerance is 0.01 in each requested field's declared units; labels and booleans require exact agreement.
Keep that rule fixed before examining a response. Inspect the explanation separately for unsupported claims.

**Visual reflection:** point to one map pattern, cite the numerical check that supports it, and name one thing
the visual cannot establish. Re-run one parameter change and preserve both maps. The implementation above
is available for inspection and modification; it is not a measured claim about model accuracy.


In [ ]:
from experiments import grade_answer
extension_source='NOT RUN'
extension_model=''; extension_date=''; extension_interface=''; extension_condition='text-and-map'
extension_response_file=''; extension_response_text=''
if extension_response_file:
    extension_response_text=Path(extension_response_file).read_text(encoding='utf-8')
if extension_source=='MODEL':
    assert extension_model and extension_date and extension_interface and extension_response_text, 'Record the actual model, date, interface and response'
extension_tolerances={key:.01 for key,value in extension_reference.items() if type(value) in (int,float)}
extension_grade=grade_answer(extension_response_text,extension_reference,extension_tolerances) if extension_response_text else dict(status='NOT RUN')
display(extension_grade)
export_json('03_extension_trial.json',dict(source=extension_source,model=extension_model,date=extension_date,
    interface=extension_interface,condition=extension_condition,prompt=extension_prompt,response_text=extension_response_text,
    tolerances=extension_tolerances,reference=extension_reference,grade=extension_grade))


## Save the investigation
Download the edited notebook, evidence, map and trial record from `exports/`. Browser storage does not
commit changes to GitHub. Keep data attribution and the distinction between observed data, synthetic
assumptions and actual model responses when sharing. The extended code and its checks above are part of
this notebook; use them as the starting point for your next controlled comparison.
